# HotpotQA — Inference, Evaluation & Contrastive Training

This notebook runs the full HalluLens pipeline on HotpotQA in **closed-book mode**
(no supporting passages provided — the model relies on parametric memory).

**Dataset:** [HotpotQA](https://huggingface.co/datasets/hotpot_qa) — multi-hop QA requiring
reasoning across two Wikipedia articles per question.
- `validation` split: 7,405 questions (default — good for eval and training the contrastive model)
- `train` split: 90,447 questions (available for large-scale runs)
- Two question types: **bridge** (chain two entities) and **comparison** (compare attributes)

**Why multi-hop matters for hallucination detection:**  
Answering correctly requires chaining two facts. Hallucinations here reflect a different
failure mode (incomplete multi-step reasoning) compared to TriviaQA / NQ (single-fact recall).
Generalisation across these two regimes is a strong paper result.

**Steps:**
1. Inference — generate model responses with activation logging
2. Evaluation — substring match against gold answer; write `eval_results.json`
3. Inspection — per-sample view, breakdown by question type and difficulty
4. Training — contrastive model on HotpotQA activations
5. OOD evaluation — score the trained model

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 — Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# ── Repo root on path ──────────────────────────────────────────────────────
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# ── Model ──────────────────────────────────────────────────────────────────
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# ── Dataset split and sample count ────────────────────────────────────────
# "validation" (7,405 questions) is the default and is large enough for
# contrastive training — ActivationParser handles the train/test split internally.
# Switch to "train" only if you need the full 90k for scale experiments.
SPLIT = "validation"   # "validation" | "train"
N     = None            # None = entire split

# ── Storage paths ──────────────────────────────────────────────────────────
base_dir          = Path("shared/hotpotqa")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "hotpotqa" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file = str(task_dir / "eval_results.json")

# ── Activation logging ─────────────────────────────────────────────────────
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# ── Inference settings ─────────────────────────────────────────────────────
MAX_TOKENS  = 128
TEMPERATURE = 0.0

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'})")
print(f"Generations file  : {generations_file}")
print(f"Eval results      : {eval_results_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 — Inference

Generates model responses for HotpotQA questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="hotpotqa",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
)

## 3 — Evaluation

Scores each generation against the gold answer (case-insensitive substring match).
Writes `eval_results.json` (ActivationParser-compatible) and `raw_eval_res.jsonl`.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="eval",
    task="hotpotqa",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

## 4 — Results inspection

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    eval_res = json.load(f)

print(f"Model        : {model_name}")
print(f"Split        : {SPLIT}")
print(f"Total        : {eval_res['total_count']}")
print(f"Correct      : {eval_res['accurate_count']}  ({eval_res['correct_rate']:.1%})")
print(f"Hallucinated : {eval_res['hallu_count']}  ({eval_res['halu_Rate']:.1%})")

In [ ]:
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)
print(f"Loaded {len(raw_df)} samples")
raw_df.head()

In [ ]:
# ── Hallucination rate by question type (bridge vs comparison) ─────────────
if "type" in raw_df.columns:
    type_stats = (
        raw_df.groupby("type")["is_hallucination"]
        .agg(halu_rate="mean", count="count")
        .reset_index()
    )
    type_stats["halu_rate"] = type_stats["halu_rate"].map("{:.1%}".format)
    print("By question type:")
    display(type_stats)

# ── Hallucination rate by difficulty level ─────────────────────────────────
if "level" in raw_df.columns:
    level_stats = (
        raw_df.groupby("level")["is_hallucination"]
        .agg(halu_rate="mean", count="count")
        .reset_index()
    )
    level_stats["halu_rate"] = level_stats["halu_rate"].map("{:.1%}".format)
    print("\nBy difficulty level:")
    display(level_stats)

In [ ]:
# ── Browse hallucinated examples ───────────────────────────────────────────
halu_df = raw_df[raw_df["is_hallucination"]].reset_index(drop=True)
print(f"Hallucinated: {len(halu_df)}")

for i, row in halu_df.head(5).iterrows():
    print(f"\n--- [{row.get('type', '')} / {row.get('level', '')}] ---")
    print(f"Q  : {row['question']}")
    print(f"Gen: {row['generation']}")
    print(f"Ans: {row['answer']}")

## 5 — Contrastive Training

Trains a `ProgressiveCompressor` on HotpotQA activations.
Requires `activations_path` to exist (logged during inference).

In [ ]:
import torch
from loguru import logger as _logger
_logger.remove()
_logger.add(sys.stdout, level="INFO")

from activation_logging.activation_parser import ActivationParser
from activation_research.model import ProgressiveCompressor
from activation_research.training import train_contrastive
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator
from torch.utils.data import DataLoader

_act = Path(activations_path)
if not _act.exists():
    raise FileNotFoundError(
        f"Activations not found: {activations_path}\n"
        "Re-run inference with activation logging enabled."
    )
print(f"Activations found: {_act.resolve()}")

In [ ]:
# ── Training config ────────────────────────────────────────────────────────
relevant_layers = list(range(14, 30))
target_layers   = [22, 26]
num_views       = 4
outlier_class   = 1

input_dim      = 4096
final_dim      = 512
max_epochs     = 50
batch_size     = 512
sub_batch_size = 64
lr             = 1e-5
temperature    = 0.25
num_workers    = 4

checkpoint_dir = "checkpoints/contrastive_hotpotqa"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# ── Load datasets ──────────────────────────────────────────────────────────
ap = ActivationParser(
    inference_json=generations_file,
    eval_json=eval_results_file,
    activations_path=activations_path,
    logger_type=LOGGER_TYPE,
    verbose=False,
)

train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)
test_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    num_views=num_views,
    backend=LOGGER_TYPE,
    preload=True,
)

print(f"Train: {len(train_dataset)}  |  Test: {len(test_dataset)}")
print(ap.df["halu"].value_counts(dropna=False).rename(index={0: "non-halu", 1: "halu"}))

In [ ]:
# ── Build and train model ──────────────────────────────────────────────────
model = ProgressiveCompressor(
    input_dim=input_dim,
    final_dim=final_dim,
    input_dropout=0.3,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

train_contrastive(
    model,
    train_dataset,
    test_dataset=test_dataset,
    epochs=max_epochs,
    batch_size=batch_size,
    sub_batch_size=sub_batch_size,
    lr=lr,
    temperature=temperature,
    device=device,
    num_workers=num_workers,
    persistent_workers=num_workers > 0,
    use_labels=True,
    ignore_label=outlier_class,
    checkpoint_dir=checkpoint_dir,
    snapshot_every=10,
    snapshot_keep_last=5,
)

## 6 — OOD Evaluation

In [ ]:
train_ds_eval = train_dataset.slice_layers(target_layers)
test_ds_eval  = test_dataset.slice_layers(target_layers)

train_loader = DataLoader(train_ds_eval, batch_size=64, shuffle=False)
eval_loader  = DataLoader(test_ds_eval,  batch_size=64, shuffle=False)

model.eval().to(device)

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=device,
    num_workers=num_workers,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        "cosine",
        "mds",
        {
            "metric": "knn",
            "kwargs": {
                "k": 50,
                "metric": "euclidean",
                "calibrate_k": True,
                "k_candidates": [50, 100, 200, 500],
                "max_train_size": 200000,
                "sample_seed": 0,
            },
            "train_selection": "all",
        },
    ],
)

ood_stats = ood_eval.compute(eval_loader, model)
print("OOD metrics (HotpotQA):")
for k, v in ood_stats.items():
    print(f"  {k}: {v:.4f}")